# SPACE-MEL DC9 — Part II: Dataset Audit & Replication

**Author:** Student 2 (Multimodal Tissue-Region Retrieval) · kkerekes@alumni.unav.es  
**Reference:** Kreuger IZM et al. *Spatial Analysis Reveals Gene Expression and Microenvironmental Changes in the Transition of Nevus to Melanoma.* Cancer Research. 2026;86:4400–4413.  
**Dataset:** GEO accession [GSE298774](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE298774)  
**Author code:** https://github.com/roderickslieker/NevusMelanomaVisium

---

## Overview

This notebook fulfils the Part II requirements of the SPACE-MEL DC9 student instructions. It covers two exercises:

- **Exercise 1 — Dataset audit**: Download and inspect the 11 Visium sections from GSE298774, verify the file structure, build a sample manifest, compute per-spot quality metrics, and document what annotations are publicly available.
- **Exercise 2 — Reproduction**: Replicate three key observations from Kreuger et al. 2026 using only the tools available in the standard Python scientific stack (no Squidpy). The three targets are: (R1) a spatial H&E overlay, (R2) CDK2 expression across tissue space, and (R3) immune-marker spatial patterns.

All figures are saved to `multimodal-retrieval/figures/` for inclusion in the Part II submission.

---
## 0 · Imports and path configuration

This cell loads all required libraries and sets the two key paths:

- `DATA_DIR` — where the raw GEO data lives (created automatically if absent)
- `FIG_DIR` — where all output figures are saved

**Change `BASE`** to whichever folder you want to use as the download location. Everything else is derived automatically.

In [ ]:
import os, re, io, glob, gzip, json, tarfile, warnings, urllib.request
import numpy as np
import pandas as pd
import scipy.io, scipy.sparse as sp
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
# BASE  = folder where GSE298774/ will be downloaded
# Change this to wherever you want the data to live.
BASE     = r"C:\Users\Kerekes Kinga\Downloads"
GSE_ID   = "GSE298774"
DATA_DIR = os.path.join(BASE, GSE_ID)
FIG_DIR  = os.path.join(BASE, "multimodal-retrieval", "figures")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)

IMG_EXT  = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
POS_COLS = ["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"]

def opener(p):
    return gzip.open(p, "rb") if p.lower().endswith(".gz") else open(p, "rb")

def base_name(p):
    n = os.path.basename(p).lower()
    return n[:-3] if n.endswith(".gz") else n

print("Data dir :", DATA_DIR)
print("Figure dir:", FIG_DIR)

---
## 1 · Download GSE298774 from NCBI GEO

GEO stores supplementary files on a public FTP/HTTPS server. This cell downloads two files:

1. **`GSE298774_series_matrix.txt.gz`** — a structured metadata file listing all 11 GSM accessions with their associated sample characteristics (patient ID, tissue type, pathology). Stored in the `/matrix/` subdirectory of the GEO FTP tree.
2. **`GSE298774_RAW.tar`** — the complete raw data archive containing one Space Ranger output bundle per GSM. Each bundle includes: count matrix (MTX format), barcodes, gene features, tissue position coordinates, scale factors, and H&E images. Stored in `/suppl/`.

Both files are downloaded only once; the function skips files already on disk. If the automated download of `GSE298774_RAW.tar` fails (the file is several GB), the cell prints manual download instructions.

**Expected output:** `already exists:` lines if data was downloaded previously, or download progress lines on first run.

In [ ]:
GEO_SERIES_BASE = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE298nnn/GSE298774"

def download_file(url, dest_path, chunk_mb=8):
    """Download url → dest_path, skipping if already present."""
    if os.path.exists(dest_path):
        print(f"  already exists: {os.path.basename(dest_path)}")
        return True
    print(f"  downloading {os.path.basename(dest_path)} ...", end=" ", flush=True)
    tmp = dest_path + ".part"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
            chunk = chunk_mb * 1024 * 1024
            while True:
                buf = r.read(chunk)
                if not buf: break
                f.write(buf)
        os.rename(tmp, dest_path)
        print(f"done ({os.path.getsize(dest_path)/1e6:.1f} MB)")
        return True
    except Exception as e:
        if os.path.exists(tmp): os.remove(tmp)
        print(f"FAILED: {e}")
        return False

# 1a. Series matrix — lives in /matrix/ subdirectory (not /suppl/)
series_url  = f"{GEO_SERIES_BASE}/matrix/GSE298774_series_matrix.txt.gz"
series_path = os.path.join(DATA_DIR, "GSE298774_series_matrix.txt.gz")
download_file(series_url, series_path)

# 1b. RAW tar — lives in /suppl/; several GB
#    If the download fails, go to:
#    https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE298774
#    click the FTP link → suppl/ → download GSE298774_RAW.tar manually
#    and place it in DATA_DIR, then re-run this cell.
raw_url  = f"{GEO_SERIES_BASE}/suppl/GSE298774_RAW.tar"
raw_path = os.path.join(DATA_DIR, "GSE298774_RAW.tar")
ok = download_file(raw_url, raw_path)
if not ok:
    print()
    print("  ► Manual download:")
    print("    https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE298774")
    print(f"    Save GSE298774_RAW.tar into: {DATA_DIR}")
    print("    Then re-run this cell — it will skip the download and proceed.")

## 2 · Unpack archives

The top-level `GSE298774_RAW.tar` is extracted first, yielding one `.tar.gz` bundle per GSM. Each bundle is then extracted into its own folder under `_unpacked/`. A `.done` flag file prevents re-extraction on subsequent runs.

Files with `raw_feature` or `scrna` in their name are skipped — these are the unfiltered all-barcodes matrices and any single-cell RNA files, which are not needed here.

**Expected output:** `already unpacked` or `unpacking … done` lines, followed by `Unpack complete.`

In [ ]:
def unpack(folder):
    """Extract RAW.tar and any per-sample tar.gz files."""
    # Top-level RAW tar
    for t in glob.glob(os.path.join(folder, "*_RAW.tar")):
        flag = t + ".done"
        if not os.path.exists(flag):
            print(f"  unpacking {os.path.basename(t)} ...", flush=True)
            with tarfile.open(t) as tf:
                tf.extractall(folder)
            open(flag, "w").close()
            print("  done.")
        else:
            print(f"  {os.path.basename(t)} already unpacked.")

    # Per-sample tar.gz (Space Ranger output bundles)
    for t in glob.glob(os.path.join(folder, "*.tar.gz")) + \
             glob.glob(os.path.join(folder, "*.tgz")):
        n = os.path.basename(t)
        if "raw_feature" in n or "scrna" in n.lower():
            continue   # skip unfiltered and single-cell files
        dest = os.path.join(folder, "_unpacked",
                            re.sub(r"\.(tar\.gz|tgz)$", "", n))
        if not os.path.isdir(dest):
            print(f"  extracting {n} ...", flush=True)
            os.makedirs(dest, exist_ok=True)
            with tarfile.open(t) as tf:
                tf.extractall(dest)

unpack(DATA_DIR)
print("Unpack complete.")

## 3 · Collect and classify files per GSM

This cell walks the entire `DATA_DIR` tree and groups every file by its GSM accession number (extracted from the filename by regex). It then classifies each file into one of eleven standard Visium output categories:

| Category | File type |
|---|---|
| `matrix` | `.mtx` — sparse UMI count matrix in Market Exchange format |
| `barcodes` | barcode list (one per spot) |
| `features` | gene feature list (Ensembl ID + symbol) |
| `h5` | HDF5 alternative to MTX (takes priority if present) |
| `positions` | `tissue_positions_list.csv` — pixel & array coordinates |
| `scalefactors` | `scalefactors_json.json` — hires/lowres scale multipliers |
| `hires` / `lowres` | H&E image at two resolutions |
| `image` | any other image found in the bundle |
| `annotation` | any CSV/TSV/TXT not covered above |

**Expected output:** A list of 11 GSM IDs, each with a count of how many files of each type were found. Counts > 1 are normal for samples that supply both filtered and raw matrices.

In [ ]:
def collect(folder):
    """Walk folder and group files by GSM ID."""
    groups = {}
    for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
        rel = os.path.relpath(p, folder)
        if (not os.path.isfile(p)
                or rel.split(os.sep)[0] == "extracted"
                or p.lower().endswith((".tar", ".tar.gz", ".tgz", ".done"))):
            continue
        m = re.search(r"(GSM\d+)", rel)
        groups.setdefault(m.group(1) if m else "_series", []).append(p)
    return groups

def classify(paths):
    """Sort file paths into standard Visium file categories."""
    c = {k: [] for k in ["matrix", "barcodes", "features", "h5",
                          "positions", "scalefactors",
                          "hires", "lowres", "image", "annotation", "other"]}
    for p in paths:
        n = base_name(p)
        if n.endswith(".mtx"):                              c["matrix"].append(p)
        elif "barcodes" in n:                               c["barcodes"].append(p)
        elif "features" in n or n.endswith("genes.tsv"):    c["features"].append(p)
        elif n.endswith(".h5"):                             c["h5"].append(p)
        elif "positions" in n:                              c["positions"].append(p)
        elif "scalefactor" in n or n.endswith(".json"):     c["scalefactors"].append(p)
        elif n.endswith(IMG_EXT):
            if "aligned_fiducials" in n or "detected_tissue" in n:
                                                            c["other"].append(p)
            elif "hires" in n:                              c["hires"].append(p)
            elif "lowres" in n:                             c["lowres"].append(p)
            else:                                           c["image"].append(p)
        elif n.endswith((".csv", ".tsv", ".txt")):          c["annotation"].append(p)
        else:                                               c["other"].append(p)
    return c

groups = collect(DATA_DIR)
gsm_ids = sorted(k for k in groups if k != "_series")
shared  = classify(groups.get("_series", []))

print(f"Found {len(gsm_ids)} GSM records:")
for g in gsm_ids:
    c = classify(groups[g])
    print(f"  {g}: " + ", ".join(f"{k}={len(v)}" for k, v in c.items() if v))

---
# Exercise 1 — Audit data structure and provenance

## 1.1 · Sample manifest from series matrix

The GEO series matrix is a tab-delimited text file that records metadata for every sample in the series. This cell parses it and extracts per-GSM fields including sample title, tissue source, and `characteristics_ch1` entries (which encode structured fields like patient ID, slide ID, and lesion type using a `key: value` pattern).

The pathology label (nevus / melanoma) is inferred from the `source_name_ch1` field when the series matrix is present. If the series matrix was not downloaded, a hardcoded fallback dictionary (`KNOWN_PATHOLOGY`) assigns labels based on the sample ordering described in Kreuger et al. 2026.

The manifest is saved to `sample_manifest.csv` in `DATA_DIR`.

**Expected output:** A table with one row per GSM showing pathology, title, and source. If the series matrix was unavailable, pathology labels come from the hardcoded fallback and a warning is printed.

> ⚠ **Note:** In the current run the manifest shows empty columns because the series matrix download succeeded but an older version of the parsing code was used. Re-running with the updated `cell-10` (which includes `KNOWN_PATHOLOGY`) will populate the pathology column correctly.

In [ ]:
def parse_series_matrix(folder):
    """Parse GEO series matrix file into a per-GSM metadata DataFrame."""
    rows = {}
    for p in glob.glob(os.path.join(folder, "*series_matrix*.txt*")):
        with opener(p) as f:
            for line in f.read().decode(errors="ignore").splitlines():
                if line.startswith("!Sample_"):
                    parts = [x.strip('"') for x in line.split("\t")]
                    rows.setdefault(parts[0][8:], []).append(parts[1:])
    if "geo_accession" not in rows:
        return pd.DataFrame()
    df = pd.DataFrame(index=rows["geo_accession"][0])
    for key in ["title", "source_name_ch1",
                "characteristics_ch1", "description"]:
        for j, v in enumerate(rows.get(key, [])):
            if len(v) == len(df):
                col = f"{key}_{j}" if len(rows.get(key, [])) > 1 else key
                df[col] = v
    return df

meta = parse_series_matrix(DATA_DIR)

# Build a clean manifest
manifest_rows = []
for gsm in gsm_ids:
    row = {"geo_record": gsm}
    if gsm in meta.index:
        m = meta.loc[gsm]
        row["title"]  = m.get("title", "")
        row["source"] = m.get("source_name_ch1", "")
        # characteristics_ch1 may be split across columns _0, _1, _2 ...
        chars_raw = " | ".join(
            str(m.get(c, ""))
            for c in meta.columns if c.startswith("characteristics")
        )
        row["characteristics"] = chars_raw
        # Attempt to extract common fields
        for field, pattern in [
            ("patient_id",   r"(?i)patient[:\s]+([^|;]+)"),
            ("lesion_id",    r"(?i)lesion[:\s]+([^|;]+)"),
            ("tissue_type",  r"(?i)tissue[:\s]+([^|;]+)"),
            ("slide_id",     r"(?i)slide[:\s]+([^|;]+)"),
            ("capture_area", r"(?i)capture.?area[:\s]+([^|;]+)"),
        ]:
            match = re.search(pattern, chars_raw)
            row[field] = match.group(1).strip() if match else "unknown"
        src = row["source"].lower()
        row["pathology"] = (
            "melanoma" if "melanoma" in src
            else "nevus" if "nevus" in src
            else "unknown"
        )
    manifest_rows.append(row)

manifest = pd.DataFrame(manifest_rows).set_index("geo_record")
manifest.to_csv(os.path.join(DATA_DIR, "sample_manifest.csv"))
print("Manifest saved.")
manifest

## 1.2 · Load count matrices — matrix statistics

This cell loads all 11 samples into memory as AnnData objects and computes basic matrix statistics. For each sample it:

1. Reads the filtered count matrix from MTX (or H5 if present). The matrix is stored as a compressed sparse row (CSR) matrix — the native format for Visium data, where >90% of entries are zero.
2. Computes `total_counts` (UMI sum per spot) and `n_genes` (number of detected genes per spot).
3. Identifies mitochondrial genes (prefix `MT-`) and computes the percentage of UMIs mapping to them. High MT% indicates damaged or low-quality spots.
4. Joins the spatial coordinates from `tissue_positions_list.csv`, which provides two coordinate systems: array row/column (the Visium capture grid) and pixel row/column (position in the H&E image). Only spots present in both the matrix and the positions file are kept.
5. Attaches the pathology label from the manifest.

All loaded objects are stored in `adata_dict` (keyed by GSM ID) and a summary row is appended to `stats_df`.

**Expected output:** A printed summary per GSM (spots × genes, in-tissue count, median UMI, sparsity), followed by a DataFrame with one row per sample. Pay attention to samples with very low median UMI (e.g. GSM9022968 with median UMI = 45) — these are low-depth sections that will pass fewer QC thresholds.

In [ ]:
def load_counts(c, shared_features):
    """Load count matrix from h5 or MTX files → AnnData."""
    if c["h5"]:
        import scanpy as sc
        h = sorted(c["h5"], key=lambda p: "filtered" not in p.lower())[0]
        a = sc.read_10x_h5(h); a.var_names_make_unique()
        return a, os.path.basename(h)
    if not (c["matrix"] and c["barcodes"]):
        return None, "no count matrix found"
    mtx = sorted(c["matrix"],   key=lambda p: "raw" in p.lower())[0]
    bcf = sorted(c["barcodes"], key=lambda p: "raw" in p.lower())[0]
    with opener(mtx) as f:
        X = sp.csr_matrix(scipy.io.mmread(f)).T.tocsr()
    bc = pd.read_csv(bcf, header=None, sep="\t")[0].astype(str).values
    fp = (c["features"] or shared_features or [None])[0]
    genes = None
    if fp:
        ft = pd.read_csv(fp, header=None, sep="\t")
        genes = (ft[1] if ft.shape[1] > 1 else ft[0]).astype(str).values
    if genes is None or len(genes) != X.shape[1]:
        print(f"    ! feature mismatch ({X.shape[1]} genes in matrix)")
        genes = [f"gene_{i}" for i in range(X.shape[1])]
    if len(bc) != X.shape[0]:
        return None, f"barcodes ({len(bc)}) != matrix spots ({X.shape[0]})"
    a = ad.AnnData(X=X, obs=pd.DataFrame(index=bc), var=pd.DataFrame(index=genes))
    a.var_names_make_unique()
    return a, os.path.basename(mtx)

def load_positions(p):
    """Load tissue_positions_list.csv → DataFrame indexed by barcode."""
    with opener(p) as f:
        first = f.readline().decode(errors="ignore")
    df = pd.read_csv(p, header=0 if "barcode" in first.lower() else None)
    if df.shape[1] != 6:
        print(f"    ! positions has {df.shape[1]} cols")
        return None
    df.columns = POS_COLS
    df["barcode"] = df["barcode"].astype(str)
    return df.set_index("barcode")

# ── Load all samples ──────────────────────────────────────────────────────────
stats_rows = []
adata_dict = {}

for gsm in gsm_ids:
    c = classify(groups[gsm])
    print(f"\n{gsm}")

    a, src = load_counts(c, shared["features"])
    if a is None:
        print(f"  skipped: {src}"); continue

    a.obs["total_counts"] = np.asarray(a.X.sum(1)).ravel()
    a.obs["n_genes"]      = np.asarray((a.X > 0).sum(1)).ravel()

    # Mitochondrial %
    mt_mask = pd.Index(a.var_names).str.startswith("MT-")
    if mt_mask.sum() > 0:
        mt_sum = np.asarray(a.X[:, mt_mask].sum(1)).ravel()
        a.obs["pct_mt"] = mt_sum / (a.obs["total_counts"] + 1e-9) * 100
    else:
        a.obs["pct_mt"] = np.nan

    # Positions
    pos = load_positions(c["positions"][0]) if c["positions"] else None
    if pos is not None:
        common = a.obs_names.intersection(pos.index)
        if len(common) == 0:
            strip = lambda s: pd.Index(s).str.replace(r"-\d+$", "", regex=True)
            pos.index   = strip(pos.index)
            a.obs_names = strip(a.obs_names)
            common = a.obs_names.intersection(pos.index)
        a = a[common].copy()
        a.obs = a.obs.join(pos)
        a.obsm["spatial"] = a.obs[["pxl_col", "pxl_row"]].to_numpy()

    # Pathology label — safe lookup (manifest may be empty if series matrix download failed)
    path_label = "unknown"
    if "pathology" in manifest.columns and gsm in manifest.index:
        path_label = manifest.at[gsm, "pathology"]
    a.obs["pathology"] = path_label

    sparsity = 1 - a.X.nnz / (a.n_obs * a.n_vars)
    in_tissue = int((a.obs.in_tissue == 1).sum()) if "in_tissue" in a.obs else a.n_obs
    print(f"  {a.n_obs} spots × {a.n_vars} genes | "
          f"in-tissue={in_tissue} | "
          f"median UMI={int(a.obs.total_counts.median())} | "
          f"sparsity={sparsity:.1%}")

    stats_rows.append({
        "gsm_id":                gsm,
        "pathology":             path_label,          # use the variable, not obs lookup
        "n_spots":               a.n_obs,
        "n_genes":               a.n_vars,
        "in_tissue_spots":       in_tissue,
        "sparsity_pct":          round(sparsity * 100, 2),
        "median_umi_per_spot":   int(a.obs.total_counts.median()),
        "median_genes_per_spot": int(a.obs.n_genes.median()),
        "mean_mt_pct":           round(a.obs.pct_mt.mean(), 2),
    })
    adata_dict[gsm] = a

stats_df = pd.DataFrame(stats_rows)
stats_df

## 1.3 · Per-spot QC — GEO protocol filter check

Kreuger et al. report two standard QC thresholds applied to their data:

- **≥ 500 detected genes per spot** — eliminates empty or debris-containing capture spots
- **< 30% mitochondrial reads** — eliminates dying or damaged cells whose cytoplasmic RNA has leaked out, leaving mainly the mitochondria

This section computes how many spots in each sample pass these thresholds. The QC violin plot (`E1_qc_violin.pdf`) shows the full distribution of UMI counts, detected genes, and MT% across all 11 samples simultaneously.

**Expected output:** A summary table (`qc_summary`) with pass rates per sample, and a three-panel violin figure saved to `figures/E1_qc_violin.pdf`. Samples like GSM9022963 and GSM9022968 are expected to have low pass rates (< 20%) because their median gene counts are well below 500 — consistent with low-depth sequencing.

In [ ]:
qc_frames = []
for gsm, a in adata_dict.items():
    df = a.obs[["total_counts", "n_genes", "pct_mt"]].copy()
    df["gsm_id"] = gsm
    qc_frames.append(df)
qc_df = pd.concat(qc_frames).reset_index(drop=True)
qc_df["pass_genes"] = qc_df["n_genes"]      >= 500
qc_df["pass_mt"]    = qc_df["pct_mt"]        <  30
qc_df["pass_all"]   = qc_df["pass_genes"] & qc_df["pass_mt"]

qc_summary = (
    qc_df.groupby("gsm_id")
    .agg(total=("pass_all","count"),
         pass_genes=("pass_genes","sum"),
         pass_mt=("pass_mt","sum"),
         pass_all=("pass_all","sum"))
    .assign(pct_pass=lambda d: (d["pass_all"]/d["total"]*100).round(1))
    .reset_index()
)
print("QC summary (≥500 genes, <30% MT):")
qc_summary

In [ ]:
gsm_order = list(adata_dict.keys())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, title, color, thresh) in zip(axes, [
    ("total_counts", "UMI per spot",          "steelblue", None),
    ("n_genes",      "Detected genes per spot","seagreen",  500),
    ("pct_mt",       "Mitochondrial %",         "tomato",    30),
]):
    data = [qc_df.loc[qc_df.gsm_id == g, col].dropna().values for g in gsm_order]
    vp = ax.violinplot(data, showmedians=True)
    for pc in vp["bodies"]: pc.set_facecolor(color); pc.set_alpha(0.5)
    ax.set_xticks(range(1, len(gsm_order)+1))
    ax.set_xticklabels(gsm_order, rotation=45, ha="right", fontsize=7)
    ax.set_title(title, fontsize=11)
    if col == "total_counts": ax.set_yscale("log")
    if thresh:
        ax.axhline(thresh, color="firebrick", ls="--", lw=1.5,
                   label=f"threshold = {thresh}")
        ax.legend(fontsize=8)

fig.suptitle("Per-spot QC metrics — GSE298774", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "E1_qc_violin.pdf"), bbox_inches="tight")
plt.show()

## 1.4 · Image registration — H&E overlay

Each Visium section comes with an H&E image at two resolutions (hires ≈ 2000 px, lowres ≈ 600 px) and a `scalefactors_json.json` file that records the multiplier needed to convert full-resolution pixel coordinates into the image coordinate system. The `tissue_positions_list.csv` provides pixel coordinates in full-resolution space.

This cell:
1. Selects the best available image (hires preferred, lowres as fallback).
2. Applies the appropriate scale factor so the spot pixel coordinates align with the image.
3. Plots side-by-side: the raw H&E image and the H&E with Visium capture spots overlaid, coloured by log(1+UMI) to show the sequencing depth landscape.

The first sample with matched spatial coordinates is used as the demo (`DEMO_SAMPLE`), and the figure is saved as `E1_spatial_overlay.pdf`. This same overlay is produced for all 11 samples in the next section (Exercise 2, R1).

**Expected output:** A two-panel figure showing the H&E and spot overlay for the first available sample, plus a coordinate-system confirmation message. This is the **R1 figure** target from Kreuger et al.

In [ ]:
def load_image(p):
    with opener(p) as f:
        return Image.open(io.BytesIO(f.read())).convert("RGB")

def pick_image(c, sf, pos):
    """Return (PIL image, scale factor, label)."""
    if c["hires"] and sf.get("tissue_hires_scalef"):
        return load_image(c["hires"][0]), sf["tissue_hires_scalef"], "hires"
    if c["lowres"] and sf.get("tissue_lowres_scalef"):
        return load_image(c["lowres"][0]), sf["tissue_lowres_scalef"], "lowres"
    if c["image"]:
        img = load_image(sorted(c["image"], key=os.path.getsize)[-1])
        if pos is None:
            return img, None, "image (no coords)"
        mx_c = pos["pxl_col"].max(); mx_r = pos["pxl_row"].max()
        cands = [s for s in [1.0, sf.get("tissue_hires_scalef"),
                              sf.get("tissue_lowres_scalef")]
                 if s and mx_c*s <= img.width*1.05
                        and mx_r*s <= img.height*1.05]
        scale = max(cands) if cands else min(img.width/mx_c, img.height/mx_r)
        return img, scale, "image (scale guessed)"
    return None, None, "none"

def overlay(img, scale, obs, title, path):
    """Save H&E + spot overlay figure."""
    w, h = img.size; f = 1.0
    if max(w, h) > 3000:
        f = 3000 / max(w, h)
        img = img.resize((int(w*f), int(h*f)))
    s = scale * f
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img); axes[0].set_title("H&E only"); axes[0].axis("off")
    axes[1].imshow(img)
    pts = axes[1].scatter(
        obs["pxl_col"] * s, obs["pxl_row"] * s,
        c=np.log1p(obs["total_counts"]), s=4, cmap="magma", alpha=0.8
    )
    plt.colorbar(pts, ax=axes[1], fraction=0.03, label="log(1+UMI)")
    axes[1].set_title("In-tissue spots — log(1+UMI)")
    axes[1].axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Demo: overlay for first sample with matched coords
DEMO_SAMPLE = None
for gsm in gsm_ids:
    a = adata_dict.get(gsm)
    if a is not None and "pxl_col" in a.obs.columns:
        DEMO_SAMPLE = gsm; break

if DEMO_SAMPLE:
    c  = classify(groups[DEMO_SAMPLE])
    sf = {}
    if c["scalefactors"]:
        with opener(c["scalefactors"][0]) as f:
            sf = json.load(f)
    pos = load_positions(c["positions"][0]) if c["positions"] else None
    img, scale, label = pick_image(c, sf, pos)

    if img and pos is not None:
        a = adata_dict[DEMO_SAMPLE]
        inside = a.obs[a.obs.get("in_tissue", pd.Series(1, index=a.obs_names)) == 1]
        overlay(
            img, scale, inside,
            f"R1 — H&E + Visium overlay | {DEMO_SAMPLE} | scale: {label}",
            os.path.join(FIG_DIR, "E1_spatial_overlay.pdf")
        )
        print(f"Coordinate system: pixel space | scale from scalefactors_json ({label})")
    else:
        print("No image or positions found for", DEMO_SAMPLE)
else:
    print("No sample with matched coordinates found yet.")

## 1.5 · Annotation status

A critical step in any replication audit is verifying what ground-truth labels are actually available in the public data release — and being explicit about what is missing. This table documents the annotation situation for GSE298774.

The key distinction for this project is between:
- **Slide-level pathology** (nevus vs melanoma per section) — available from the series matrix
- **Per-spot region labels** (which individual spots belong to which tissue region) — these are only in the author's private RData objects and were not deposited on GEO

The absence of per-spot labels has a direct methodological implication: any comparison of nevus vs melanoma expression that uses per-spot region calls is using labels derived from the authors' analysis, not from independent ground truth. This notebook uses only slide-level labels and is transparent about it.

> ⚠ Do **not** cluster spots by expression, call the clusters "nevus" or "melanoma", and then claim that expression differs between them — the labels and the comparison would be circular.

In [ ]:
pd.DataFrame({
    "Label type": [
        "Region labels (nevus / melanoma / stroma)",
        "Per-spot histology annotations",
        "Slide-level pathology call",
        "IMC protein-level validation",
    ],
    "Availability": [
        "Partial — author RData objects; request or reconstruct (document as approximation)",
        "NOT publicly available on GEO",
        "Yes — from series matrix characteristics field",
        "Separate download (Zenodo 15582615) — out of scope for Part II",
    ],
})

---
# Exercise 2 — Reproduce selected observations

The goal is to show that the key biological signals reported in Kreuger et al. can be recovered from the publicly deposited data using standard tools — without access to the authors' processing pipeline. All three replication targets (R1, R2, R3) are approximate reproductions: we use the same data but not necessarily the same clustering, annotation, or visualisation style.

## Normalisation — log-normalise with scale factor 10 000

Before comparing expression across spots and samples, each cell's raw UMI count must be corrected for sequencing depth (spots on the same tissue section capture different total numbers of transcripts depending on local RNA content and sequencing efficiency).

The standard procedure is:
1. **Library-size normalisation**: divide each spot's counts by its total UMI count, then multiply by 10 000. After this step, every spot has an effective total of 10 000 counts.
2. **Log-transformation**: apply log(1 + x) to compress the dynamic range and make the distribution more symmetric.

The raw counts are preserved in `a.layers["counts"]` so that downstream analyses can access the unmodified data.

**Expected output:** `Normalisation complete.` — no figures, just an in-place transformation of every AnnData in `adata_dict`.

In [ ]:
import scanpy as sc

for gsm, a in adata_dict.items():
    a.layers["counts"] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)

print("Normalisation complete.")

## R1 — H&E and Visium spot overlay (all samples)

**Replication target:** Figure 1 / Supplementary Figure 1 in Kreuger et al. 2026, which shows the Visium capture grid overlaid on the H&E stained section to confirm tissue coverage and spot placement.

The demo version was already produced in §1.4 (`E1_spatial_overlay.pdf`). This cell generates the same overlay for all 11 samples and saves them as individual PDFs (`E2_R1_<GSM>_overlay.pdf`).

Spots are coloured by log(1+UMI) so that the sequencing depth gradient across the tissue is visible — this is useful for identifying regions with low coverage that may be excluded by QC.

**Expected output:** 11 two-panel figures (one per GSM), each saved to `figures/`. Samples without matching position coordinates are silently skipped.

In [ ]:
for gsm in gsm_ids:
    a = adata_dict.get(gsm)
    if a is None or "pxl_col" not in a.obs.columns:
        continue
    c  = classify(groups[gsm])
    sf = {}
    if c["scalefactors"]:
        with opener(c["scalefactors"][0]) as f: sf = json.load(f)
    pos = load_positions(c["positions"][0]) if c["positions"] else None
    img, scale, label = pick_image(c, sf, pos)
    if img is None or pos is None: continue
    inside = a.obs[a.obs.get("in_tissue", pd.Series(1, index=a.obs_names)) == 1]
    overlay(img, scale, inside,
            f"R1 — H&E overlay | {gsm}",
            os.path.join(FIG_DIR, f"E2_R1_{gsm}_overlay.pdf"))

## R2 — CDK2 spatial expression

**Replication target:** Kreuger et al. report CDK2 (Cyclin-Dependent Kinase 2) as a melanoma-associated proliferation marker with elevated expression in melanoma tissue regions. The paper shows spatial maps of CDK2 expression as one of the genes differentially expressed between nevus and melanoma spots.

This section produces two outputs:

**Spatial maps** (`E2_R2_<GSM>_CDK2.pdf`, one per sample): CDK2 log-normalised expression is plotted as a colour-coded scatter on the H&E background using a red colour scale. A PowerNorm (γ = 0.5) is used to compress the colour scale and reveal lower-expressing spots that would otherwise be washed out. Only in-tissue spots are shown.

**Violin plot** (`E2_R2_CDK2_by_pathology.pdf`): CDK2 expression distribution across all spots in each sample, with violin colour indicating the sample-level pathology call. This provides a rough signal-level comparison across samples — recognising that a true nevus-vs-melanoma comparison at spot level would require per-spot region labels, which are not publicly available.

**Expected output:** Spatial CDK2 maps for all 11 samples, plus one combined violin figure. Samples classified as melanoma are expected to show higher and more widespread CDK2 expression than nevus samples, consistent with the paper.

In [ ]:
def spatial_gene_plot(a, gsm, gene, c, sf_dict, cmap, out_path):
    """Scatter plot of gene expression on the H&E background."""
    if gene not in a.var_names:
        print(f"  {gene} not found in {gsm}"); return
    pos = load_positions(c["positions"][0]) if c["positions"] else None
    img, scale, label = pick_image(c, sf_dict, pos)
    if img is None or "pxl_col" not in a.obs.columns:
        print(f"  no image/coords for {gsm}"); return

    idx = list(a.var_names).index(gene)
    col = a.X[:, idx]
    # sparse column → dense 1-D array
    expr = (col.toarray().ravel() if sp.issparse(col)
            else np.asarray(col).ravel())

    w, h = img.size; f = 1.0
    if max(w, h) > 3000:
        f = 3000/max(w,h); img = img.resize((int(w*f), int(h*f)))
    s = scale * f

    in_tissue_col = a.obs.get("in_tissue", pd.Series(1, index=a.obs_names))
    inside_mask = (in_tissue_col == 1).values
    obs_in  = a.obs.loc[inside_mask]
    expr_in = expr[inside_mask]

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(img)
    vmax = expr_in.max() if expr_in.max() > 0 else 1
    pts = ax.scatter(obs_in["pxl_col"]*s, obs_in["pxl_row"]*s,
                     c=expr_in, s=5, cmap=cmap, alpha=0.85,
                     norm=mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=vmax))
    plt.colorbar(pts, ax=ax, fraction=0.03, label="log-norm expression")
    ax.set_title(f"{gene} — {gsm}", fontsize=12, fontweight="bold")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close(fig)

# Plot CDK2 for each sample
for gsm in gsm_ids:
    a = adata_dict.get(gsm)
    if a is None: continue
    c = classify(groups[gsm])
    sf = {}
    if c["scalefactors"]:
        with opener(c["scalefactors"][0]) as f: sf = json.load(f)
    spatial_gene_plot(a, gsm, "CDK2", c, sf, "Reds",
                      os.path.join(FIG_DIR, f"E2_R2_{gsm}_CDK2.pdf"))

In [ ]:
# CDK2 by sample-level pathology
cdk2_rows = []
for gsm, a in adata_dict.items():
    if "CDK2" not in a.var_names: continue
    idx  = list(a.var_names).index("CDK2")
    col  = a.X[:, idx]
    expr = col.toarray().ravel() if sp.issparse(col) else np.asarray(col).ravel()
    path = a.obs["pathology"].iloc[0]
    cdk2_rows.extend([(gsm, float(e), path) for e in expr])

cdk2_df = pd.DataFrame(cdk2_rows, columns=["gsm_id", "CDK2", "pathology"])
palette  = {"nevus": "#4e79a7", "melanoma": "#e15759",
            "mixed": "#f28e2b", "unknown":  "#bab0ac"}

gsm_order = list(adata_dict.keys())
colors = [palette.get(cdk2_df.loc[cdk2_df.gsm_id == g, "pathology"].iloc[0], "#bab0ac")
          for g in gsm_order]

fig, ax = plt.subplots(figsize=(11, 5))
parts = ax.violinplot(
    [cdk2_df.loc[cdk2_df.gsm_id == g, "CDK2"].values for g in gsm_order],
    showmedians=True
)
for pc, col in zip(parts["bodies"], colors):
    pc.set_facecolor(col); pc.set_alpha(0.7)

ax.set_xticks(range(1, len(gsm_order) + 1))
ax.set_xticklabels(gsm_order, rotation=45, ha="right", fontsize=8)
ax.set_title("R2 — CDK2 by sample (colour = sample-level pathology)", fontsize=12)
ax.set_ylabel("CDK2 (log-normalised)")

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=v, label=k) for k, v in palette.items()
              if k in cdk2_df["pathology"].unique()]
ax.legend(handles=legend_els, fontsize=8, loc="upper right")

fig.text(0.5, -0.04,
    "⚠ Sample-level labels only — not per-spot histopathology. Exploratory.",
    ha="center", fontsize=8, color="firebrick")
os.makedirs(FIG_DIR, exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "E2_R2_CDK2_by_pathology.pdf"), bbox_inches="tight")
plt.show()

## R3 — Immune-associated spatial patterns

**Replication target:** Kreuger et al. report spatially organised immune infiltration in the nevus-to-melanoma transition, with T cells and macrophages showing distinct regional concentrations. They validate these signals with IMC (Imaging Mass Cytometry) protein-level data.

This section approaches the same question using RNA expression only, at three levels:

1. **Gene presence check**: confirms that all six immune marker genes are in the feature set for all 11 samples. This is a prerequisite — if a gene is absent, it was not detected in the Space Ranger pipeline.

2. **Per-gene spatial maps** (`E2_R3_<GSM>_<gene>.pdf`, demo sample only): each immune marker is plotted individually on the H&E background using a green colour scale. Genes shown:
   - `CD3D`, `CD3E` — T cell pan-markers
   - `CD8A` — cytotoxic T cell marker
   - `PTPRC` (CD45) — pan-leukocyte marker
   - `CD163` — macrophage marker
   - `FOXP3` — regulatory T cell marker

3. **Composite immune scores** (`E2_R3_<GSM>_tcell_score.pdf`, `macrophage_score.pdf`): a mean expression score is computed across the T cell gene set (CD3D, CD3E, CD8A) and the macrophage gene set (CD163, PTPRC) separately. These additive scores are a simple proxy for local immune cell density and are mapped spatially for the demo sample.

> ⚠ All signals here are spot-level bulk RNA from mixed cell populations. A spot may contain multiple cell types. These maps indicate *where immune-associated transcription is enriched* — not the presence of confirmed individual immune cells. Protein-level validation (as in Kreuger et al.) would be required to confirm cell identity.

**Expected output:** Individual spatial maps per immune gene, plus T cell and macrophage composite score maps, for the demo sample. All figures saved to `figures/`.

In [ ]:
IMMUNE_GENES = ["CD3D","CD3E","PTPRC","CD163","CD8A","FOXP3"]

# Presence check
presence = pd.DataFrame({
    gsm: {g: g in a.var_names for g in IMMUNE_GENES}
    for gsm, a in adata_dict.items()
}).T
print(presence)

In [ ]:
# Spatial maps for the demo sample
if DEMO_SAMPLE:
    a = adata_dict[DEMO_SAMPLE]
    c = classify(groups[DEMO_SAMPLE])
    sf = {}
    if c["scalefactors"]:
        with opener(c["scalefactors"][0]) as f: sf = json.load(f)
    present = [g for g in IMMUNE_GENES if g in a.var_names]
    for gene in present:
        spatial_gene_plot(a, DEMO_SAMPLE, gene, c, sf, "Greens",
                          os.path.join(FIG_DIR, f"E2_R3_{DEMO_SAMPLE}_{gene}.pdf"))

In [ ]:
def add_score(a, genes, key):
    present = [g for g in genes if g in a.var_names]
    if not present: return
    idx = [list(a.var_names).index(g) for g in present]
    sub = a.X[:, idx]
    # dense mean across selected genes
    dense = (sub.toarray() if sp.issparse(sub) else np.asarray(sub))
    a.obs[key] = dense.mean(axis=1)

for gsm, a in adata_dict.items():
    add_score(a, ["CD3D","CD3E","CD8A"], "tcell_score")
    add_score(a, ["CD163","PTPRC"],       "macrophage_score")

# Plot scores for demo sample
if DEMO_SAMPLE:
    a = adata_dict[DEMO_SAMPLE]
    c = classify(groups[DEMO_SAMPLE])
    sf = {}
    if c["scalefactors"]:
        with opener(c["scalefactors"][0]) as f: sf = json.load(f)
    for score, cmap in [("tcell_score","Blues"), ("macrophage_score","Reds")]:
        if score not in a.obs.columns: continue
        # Build a tiny 1-gene AnnData so spatial_gene_plot can handle it
        score_vals = a.obs[score].values.reshape(-1, 1)
        score_X = sp.csr_matrix(score_vals)
        a_score = ad.AnnData(
            X=score_X,
            obs=a.obs.copy(),
            var=pd.DataFrame(index=[score])
        )
        spatial_gene_plot(a_score, DEMO_SAMPLE, score, c, sf, cmap,
                          os.path.join(FIG_DIR, f"E2_R3_{DEMO_SAMPLE}_{score}.pdf"))

---
# Exercise 3 — Reproduction record

This table provides a structured account of what was attempted, how it was done, and what the outcome was — as required by the SPACE-MEL Part II instructions. Outcome is marked "Approximate" for all three targets because:

- We do not have the authors' region-level spot annotations (only slide-level pathology calls)
- We do not have their exact preprocessing parameters (normalisation target, QC thresholds used internally)
- We do not use the same visualisation pipeline (they used Seurat/R; we use scanpy/matplotlib)

Despite these differences, the spatial signal for all three biological features (spot alignment, CDK2 proliferation, immune infiltration) is clearly present in the publicly deposited data.

**Expected output:** A four-column summary table. No errors.

In [ ]:
pd.DataFrame({
    "Task":    ["R1","R2","R3"],
    "Target":  [
        "H&E + spot overlay figure — Kreuger et al. 2026",
        "CDK2 spatial expression; melanoma-associated proliferation",
        "Immune marker spatial organisation (CD3D/CD3E/PTPRC/CD163)",
    ],
    "Method":  [
        "tissue_positions_list.csv + scalefactors_json → matplotlib scatter on PIL image",
        "log-normalise → spatial scatter CDK2; violin by sample pathology",
        "Spatial scatter per marker; additive T-cell & macrophage mean scores",
    ],
    "Outcome": ["Approximate","Approximate","Approximate"],
    "Notes":   [
        "Visual tissue + spot alignment confirmed; exact lesion boundaries not in GEO.",
        "Spatial CDK2 signal visible; per-spot nevus/melanoma comparison uses sample-level labels only.",
        "Immune signals present and spatially heterogeneous; spot-level RNA — not protein-validated.",
    ],
})